# 02 · Training: 3 Backbones × 5 Folds

Trains the three ImageNet-pretrained backbones (EfficientNet-B4, ConvNeXt-Base, DenseNet-121) on each lesion-grouped fold.

| Setting | Value |
|---|---|
| Input | 224 × 224 RGB, ImageNet normalisation |
| Augmentation | flips, rotation, affine, colour jitter, Gaussian noise and blur, gamma, CLAHE, coarse dropout |
| Head | LayerNorm → Dropout 0.4 → Linear 512 → GELU → Dropout 0.2 → Linear 7 |
| Class imbalance | weighted random sampler + class-weighted focal loss (γ = 2.5) with label smoothing 0.1 |
| Optimiser | AdamW, lr 3 × 10⁻⁴, weight decay 10⁻⁴, batch 32, mixed precision |
| Schedule | cosine annealing with warm restarts (first cycle 10 epochs, multiplier 2, minimum 10⁻⁶) |
| Stopping | up to 30 epochs; early stopping on validation macro AUC (patience 10); best checkpoint kept |
| Seeds | 42 + fold index |

**Runtime:** several GPU hours for all 15 models. Training is **resumable** (state saved every epoch), and checkpoints that already
exist are never overwritten, so the published models stay unchanged. Set `TRAIN_VARIANT = 'ce'` and `TRAIN_FOLDS = [0]` to train the
cross-entropy models of the loss ablation.

> This notebook is a cleaned implementation of the training procedure described above. Because of GPU non-determinism, retrained
> models differ from the published checkpoints; the evaluation notebooks (03–08) use whichever checkpoints are present.

## Setup

Paths can be edited in the next cell or set as environment variables (`HAM_PROJECT_OUT`, `HAM10000_DIR`, `ISIC2019_DIR`).

In [ ]:
import os, json, math, time, random, platform, warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
try:
    from tqdm.auto import tqdm
except ImportError:                                   # progress bars are optional
    def tqdm(x, **kwargs):
        return x

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

In [ ]:
# ---------------------------------------------------------------- paths (edit here or set the environment variables)
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

def env_path(name, default):
    return Path(os.environ.get(name, str(default))).expanduser()

PROJECT_OUT  = env_path('HAM_PROJECT_OUT', REPO / 'outputs')        # caches, checkpoints, features, tables (not in Git)
HAM_DIR      = env_path('HAM10000_DIR', REPO / 'data' / 'HAM10000')  # HAM10000 images, metadata and lesion masks
ISIC2019_DIR = env_path('ISIC2019_DIR', REPO / 'data' / 'ISIC2019')  # ISIC 2019 training images, ground truth, metadata
RESULTS_DIR  = env_path('RESULTS_DIR', REPO / 'results')             # tables and figures shown in the README
SPLITS_DIR   = env_path('SPLITS_DIR', REPO / 'splits')               # published fold assignment

# ---------------------------------------------------------------- study settings (keep unchanged to reproduce the results)
CLASSES   = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
MALIGNANT = ['akiec', 'bcc', 'mel']                                  # (pre)malignant classes
MODELS    = ['efficientnet_b4', 'convnext_base', 'densenet121']
LABEL_OF  = {'efficientnet_b4': 'EfficientNet-B4', 'convnext_base': 'ConvNeXt-Base', 'densenet121': 'DenseNet-121'}
IMG_SIZE, SEED, N_FOLDS = 224, 42, 5
DROPOUT, MC_SAMPLES = 0.4, 30                                       # head dropout; MC Dropout passes per model
UNC_QUANTILE, GMM_N, N_NVREF = 0.75, 3, 1000                        # uncertainty flag; HBDS levels; nevus reference size
VARIANT = 'focal_ls'                                                # main models (focal loss + label smoothing)
EXT_MC_SEED0 = 555                                                  # MC seed offset for external data

C = len(CLASSES)
CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}
MAL_IDX = [CLASS2IDX[c] for c in MALIGNANT]
EPS = 1e-12
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DIRS = {k: PROJECT_OUT / k for k in ['splits', 'checkpoints', 'cache', 'tables', 'figures']}
for _d in list(DIRS.values()) + [RESULTS_DIR, SPLITS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def mean_sd(values, digits=4):
    s = pd.Series(values, dtype=float).dropna()
    return f'{s.mean():.{digits}f} ± {s.std(ddof=1):.{digits}f}' if len(s) > 1 else (f'{s.mean():.{digits}f}' if len(s) else 'n/a')

print(f'Python {platform.python_version()} | PyTorch {torch.__version__} | device: {DEVICE}')
print(f'Project outputs: {PROJECT_OUT}')

In [ ]:
import cv2
from PIL import Image

def find_file(root, name):
    """Locate a dataset file by name anywhere below root."""
    direct = Path(root) / name
    if direct.is_file():
        return direct
    hits = sorted(Path(root).rglob(name))
    if not hits:
        raise FileNotFoundError(f"'{name}' not found under {root}. See the Dataset section of README.md.")
    return hits[0]

def image_index(root, exclude=('segmentation', 'mask')):
    """Map image stem -> path for all images below root (lesion masks excluded)."""
    index = {}
    for p in sorted(Path(root).rglob('*')):
        if p.suffix.lower() in ('.jpg', '.jpeg', '.png') and not any(e in p.as_posix().lower() for e in exclude):
            index.setdefault(p.stem, p)
    return index

def read_rgb(path):
    with Image.open(path) as im:
        return np.array(im.convert('RGB'))

def read_rgb_224(path):
    """Pipeline image loading: PIL -> RGB -> cv2 bilinear resize to 224 x 224."""
    return cv2.resize(read_rgb(path), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

def build_image_cache(paths, npy_path, json_path, desc='caching images'):
    """Decode every image once into a uint8 array (N, 224, 224, 3). Reused if it already exists."""
    paths = [str(p) for p in paths]
    if Path(npy_path).exists() and Path(json_path).exists():
        cached = json.load(open(json_path))
        if [Path(p).stem for p in cached] == [Path(p).stem for p in paths]:
            return np.load(npy_path, mmap_mode='r')
        raise RuntimeError(f'{json_path} exists but lists different images - delete it to rebuild the cache.')
    Path(npy_path).parent.mkdir(parents=True, exist_ok=True)
    arr = np.lib.format.open_memmap(npy_path, mode='w+', dtype=np.uint8, shape=(len(paths), IMG_SIZE, IMG_SIZE, 3))
    for i, p in enumerate(tqdm(paths, desc=desc)):
        arr[i] = read_rgb_224(p)
    arr.flush(); del arr
    json.dump(paths, open(json_path, 'w'))
    return np.load(npy_path, mmap_mode='r')

def load_ham():
    """HAM10000 metadata in the order of the image cache, labels and lesion groups."""
    meta = pd.read_csv(find_file(HAM_DIR, 'HAM10000_metadata.csv')).drop_duplicates('image_id').set_index('image_id')
    order_file = DIRS['cache'] / f'images_ham10000_{IMG_SIZE}.json'
    if not order_file.exists():
        raise FileNotFoundError(f'{order_file} not found - run notebook 01 first.')
    ids = [Path(p).stem for p in json.load(open(order_file))]
    ham = meta.loc[ids].reset_index()
    return ham, ham['dx'].map(CLASS2IDX).astype(int).values, ham['lesion_id'].astype(str).values

def load_splits():
    f = DIRS['splits'] / 'cv_splits.json'
    if not f.exists():
        raise FileNotFoundError(f'{f} not found - run notebook 01 first.')
    return {int(k): {kk: np.asarray(vv, dtype=int) for kk, vv in v.items()} for k, v in json.load(open(f)).items()}

In [ ]:
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
eval_transform = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])

def make_head(in_dim, p=DROPOUT):
    """LayerNorm -> Dropout(0.4) -> Linear(512) -> GELU -> Dropout(0.2) -> Linear(7)."""
    return nn.Sequential(nn.LayerNorm(in_dim), nn.Dropout(p), nn.Linear(in_dim, 512),
                         nn.GELU(), nn.Dropout(p / 2), nn.Linear(512, C))

class SkinLesionModel(nn.Module):
    """ImageNet-pretrained timm backbone with global average pooling + classification head."""
    def __init__(self, model_name, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.feat_dim = self.backbone.num_features
        self.classifier = make_head(self.feat_dim)

    def forward_features(self, x):
        return self.backbone(x)

    def forward(self, x):
        return self.classifier(self.backbone(x))

def ckpt_path(k, m, variant=VARIANT):
    return DIRS['checkpoints'] / f'fold{k}' / variant / f'{m}.pth'

def load_model(k, m, variant=VARIANT):
    p = ckpt_path(k, m, variant)
    if not p.exists():
        raise FileNotFoundError(f'Checkpoint {p} not found - train it with notebook 02 or download the released weights.')
    ck = torch.load(p, map_location='cpu', weights_only=True)
    model = SkinLesionModel(m)
    model.load_state_dict(ck['state_dict'] if 'state_dict' in ck else ck, strict=True)
    model.to(DEVICE).eval()
    for q in model.parameters():
        q.requires_grad_(False)
    return model

INFER_BATCH = 64 if DEVICE.type == 'cuda' else 16

@torch.no_grad()
def extract_features(model, images, rows=None, batch_size=INFER_BATCH):
    """Pooled backbone features (eval mode, float16 autocast on GPU), returned as float16 as in the study."""
    rows = np.arange(len(images)) if rows is None else np.asarray(rows)
    out = []
    for s in range(0, len(rows), batch_size):
        x = torch.stack([eval_transform(image=np.ascontiguousarray(images[i]))['image'] for i in rows[s:s + batch_size]]).to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
            f = model.forward_features(x)
        out.append(f.float().cpu().numpy().astype(np.float16))
    return np.concatenate(out) if out else np.zeros((0, model.feat_dim), np.float16)

## Training configuration

In [ ]:
TRAIN_VARIANT   = os.environ.get('TRAIN_VARIANT', 'focal_ls')         # 'focal_ls' (main models) or 'ce' (loss ablation)
TRAIN_FOLDS     = [int(f) for f in os.environ.get('TRAIN_FOLDS', '0,1,2,3,4').split(',')]
TRAIN_MODELS    = os.environ.get('TRAIN_MODELS', ','.join(MODELS)).split(',')
MAX_EPOCHS      = int(os.environ.get('MAX_EPOCHS', 30))
PATIENCE        = 10
BATCH_SIZE      = int(os.environ.get('BATCH_SIZE', 32))
LR, WEIGHT_DECAY = 3e-4, 1e-4
FOCAL_GAMMA, LABEL_SMOOTHING = 2.5, 0.1
T0, T_MULT, ETA_MIN = 10, 2, 1e-6
PRETRAINED      = os.environ.get('PRETRAINED', '1') == '1'             # ImageNet weights (downloaded by timm)
NUM_WORKERS     = int(os.environ.get('NUM_WORKERS', 0 if os.name == 'nt' else 4))
USE_AMP         = DEVICE.type == 'cuda'
print({'variant': TRAIN_VARIANT, 'folds': TRAIN_FOLDS, 'models': TRAIN_MODELS, 'max epochs': MAX_EPOCHS, 'AMP': USE_AMP})

In [ ]:
ham, LABELS, GROUPS = load_ham()
SPLITS = load_splits(); FOLDS = sorted(SPLITS)
HAM_IMGS = np.load(DIRS['cache'] / f'images_ham10000_{IMG_SIZE}.npy', mmap_mode='r')
print(f'{len(ham)} images, {len(set(GROUPS))} lesions, folds {FOLDS}')

## Augmentation, sampler and loss

In [ ]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, p=0.7),
    A.Affine(translate_percent=(-0.1, 0.1), scale=(0.8, 1.2), rotate=(-15, 15), p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1, p=0.6),
    A.GaussNoise(std_range=(math.sqrt(10) / 255, math.sqrt(50) / 255), p=0.4),   # variance 10-50 on the 0-255 scale
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5),
    A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(8, 32), hole_width_range=(8, 32), p=0.5),
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

class CachedImages(torch.utils.data.Dataset):
    def __init__(self, images, rows, labels, transform):
        self.images, self.rows, self.labels, self.transform = images, np.asarray(rows), labels, transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return self.transform(image=np.ascontiguousarray(self.images[r]))['image'], int(self.labels[r])

class FocalLossLS(nn.Module):
    """Class-weighted focal loss with label smoothing: -sum_c alpha_c q_c (1 - p_c)^gamma log p_c."""
    def __init__(self, alpha, gamma=FOCAL_GAMMA, smoothing=LABEL_SMOOTHING):
        super().__init__()
        self.register_buffer('alpha', torch.as_tensor(alpha, dtype=torch.float32))
        self.gamma, self.smoothing = gamma, smoothing
    def forward(self, logits, y):
        logp = F.log_softmax(logits.float(), dim=1); p = logp.exp(); K = logits.shape[1]
        q = torch.full_like(logp, self.smoothing / (K - 1)); q.scatter_(1, y[:, None], 1 - self.smoothing)
        return -(self.alpha[None] * q * (1 - p) ** self.gamma * logp).sum(1).mean()

from sklearn.metrics import roc_auc_score, balanced_accuracy_score

def macro_auc(y, probs):
    present = [c for c in range(C) if 0 < (y == c).sum() < len(y)]
    return float(np.mean([roc_auc_score((y == c).astype(int), probs[:, c]) for c in present]))

@torch.no_grad()
def predict_probs(model, loader):
    model.eval(); out = []
    for x, _ in loader:
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            out.append(F.softmax(model(x.to(DEVICE)).float(), 1).cpu().numpy())
    return np.concatenate(out)

## Training loop

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

def train_one(k, m):
    out_dir = DIRS['checkpoints'] / f'fold{k}' / TRAIN_VARIANT; out_dir.mkdir(parents=True, exist_ok=True)
    final, last, done = out_dir / f'{m}.pth', out_dir / f'{m}_last.pth', out_dir / f'{m}.done'
    if done.exists() or (final.exists() and not last.exists()):
        print(f'fold {k} | {LABEL_OF[m]}: checkpoint exists - skipped'); return
    seed_everything(SEED + k)
    sp = SPLITS[k]; tr, va = sp['train'], sp['val']
    counts = np.bincount(LABELS[tr], minlength=C)
    sampler = WeightedRandomSampler(torch.as_tensor(1.0 / counts[LABELS[tr]], dtype=torch.double), num_samples=len(tr),
                                    replacement=True, generator=torch.Generator().manual_seed(SEED + k))
    train_loader = DataLoader(CachedImages(HAM_IMGS, tr, LABELS, train_transform), batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda', drop_last=True)
    val_loader = DataLoader(CachedImages(HAM_IMGS, va, LABELS, eval_transform), batch_size=2 * BATCH_SIZE,
                            num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')
    model = SkinLesionModel(m, pretrained=PRETRAINED).to(DEVICE)
    alpha = 2.0 * len(tr) / (C * np.maximum(counts, 1))                          # alpha_c = 2N / (K N_c)
    criterion = (FocalLossLS(alpha) if TRAIN_VARIANT == 'focal_ls' else nn.CrossEntropyLoss()).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=T0, T_mult=T_MULT, eta_min=ETA_MIN)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    start, best, wait, history = 0, -np.inf, 0, []
    if last.exists():                                                            # resume an interrupted run
        st = torch.load(last, map_location='cpu', weights_only=False)
        model.load_state_dict(st['model']); optimizer.load_state_dict(st['optimizer'])
        scheduler.load_state_dict(st['scheduler']); scaler.load_state_dict(st['scaler'])
        start, best, wait, history = st['epoch'], st['best'], st['wait'], st['history']
        print(f'fold {k} | {LABEL_OF[m]}: resuming at epoch {start + 1}')
    for epoch in range(start, MAX_EPOCHS):
        model.train(); t0 = time.time(); losses = []
        n_batches = len(train_loader)
        for i, (x, y) in enumerate(tqdm(train_loader, desc=f'fold {k} {LABEL_OF[m]} epoch {epoch + 1}', leave=False)):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
            loss = criterion(logits.float(), y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            scheduler.step(epoch + (i + 1) / n_batches)
            losses.append(loss.item())
        probs = predict_probs(model, val_loader)
        auc = macro_auc(LABELS[va], probs); ba = balanced_accuracy_score(LABELS[va], probs.argmax(1))
        history.append({'epoch': epoch + 1, 'train loss': float(np.mean(losses)), 'val macro AUC': auc, 'val balanced acc.': ba,
                        'lr': optimizer.param_groups[0]['lr'], 'seconds': round(time.time() - t0, 1)})
        improved = auc > best
        if improved:
            best, wait = auc, 0
            torch.save({'state_dict': model.state_dict(), 'epoch': epoch + 1, 'val_macro_auc': auc, 'model': m, 'fold': k,
                        'variant': TRAIN_VARIANT}, final)
        else:
            wait += 1
        print(f"fold {k} | {LABEL_OF[m]:<15} | epoch {epoch + 1:>2} | loss {history[-1]['train loss']:.4f} | "
              f"val AUC {auc:.4f} | val BA {ba:.4f}{'  *' if improved else ''}")
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                    'scaler': scaler.state_dict(), 'epoch': epoch + 1, 'best': best, 'wait': wait, 'history': history}, last)
        if wait >= PATIENCE:
            print(f'fold {k} | {LABEL_OF[m]}: early stopping'); break
    pd.DataFrame(history).to_csv(out_dir / f'{m}_history.csv', index=False)
    done.touch(); last.unlink(missing_ok=True)
    del model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

for k in TRAIN_FOLDS:
    for m in TRAIN_MODELS:
        train_one(k, m)

## Trained checkpoints

In [ ]:
rows = []
for k in FOLDS:
    for m in MODELS:
        p = DIRS['checkpoints'] / f'fold{k}' / TRAIN_VARIANT / f'{m}.pth'
        if p.exists():
            ck = torch.load(p, map_location='cpu', weights_only=False)
            rows.append({'fold': k, 'model': LABEL_OF[m], 'best epoch': ck.get('epoch'),
                         'val macro AUC': ck.get('val_macro_auc'), 'size (MB)': round(p.stat().st_size / 2 ** 20, 1)})
ckpts = pd.DataFrame(rows); display(ckpts)
if len(ckpts):
    ckpts.to_csv(DIRS['tables'] / f'checkpoints_{TRAIN_VARIANT}.csv', index=False)

In [ ]:
hist_files = sorted((DIRS['checkpoints']).glob(f'fold*/{TRAIN_VARIANT}/*_history.csv'))
if hist_files:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    for f in hist_files:
        h = pd.read_csv(f); lab = f'{f.parent.parent.name} {LABEL_OF.get(f.stem.replace("_history", ""), f.stem)}'
        axes[0].plot(h['epoch'], h['train loss'], alpha=.7, label=lab); axes[1].plot(h['epoch'], h['val macro AUC'], alpha=.7, label=lab)
    axes[0].set_title('training loss'); axes[1].set_title('validation macro AUC')
    for ax in axes:
        ax.set_xlabel('epoch'); ax.grid(alpha=.3)
    axes[1].legend(fontsize=6, ncol=2); plt.tight_layout(); plt.savefig(DIRS['figures'] / 'training_curves.png', dpi=150); plt.show()
else:
    print('No training histories in this output folder (the checkpoints were trained elsewhere).')